In [145]:
import pandas as pd
import numpy as np

In [146]:
def answer_one():
    energy = pd.read_excel('Energy Indicators.xls', skiprows = list(range(0, 17)) + list(range(245, 284)), usecols = [2, 3, 4, 5], header = None, names = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'])

    energy = energy.replace("...", np.nan).infer_objects()
    energy['Energy Supply'] = energy['Energy Supply'].copy() * 10**6
    energy['Country'] = energy['Country'].str.replace(r'\d+' , '' , regex=True)
    energy['Country'] = energy['Country'].str.replace(r'\s*\(.*\)', '', regex=True)
    energy['Country'] = energy['Country'].replace(['Republic of Korea','United States of America', 'United Kingdom of Great Britain and Northern Ireland', 'China, Hong Kong Special Administrative Region'], ['South Korea', 'United States', 'United Kingdom', 'Hong Kong'])

    GDP = pd.read_excel('world_bank.xls', header = None, skiprows = 3)
    GDP = GDP.rename(columns = {'Country Name': 'Country'})
    GDP.columns = ['Country', 'Country Code', 'Indicator Name', 'Indicator Code'] + [str(year) for year in range(1960, 2023)]
    GDP['Country'] = GDP['Country'].replace(['Korea, Rep.','Iran, Islamic Rep.', 'Hong Kong SAR, China'], ['South Korea', 'Iran','Hong Kong'])
    GDP_years = GDP.copy()[['Country'] + [str(year) for year in range(2006, 2016)]]

    ScimEn = pd.read_excel('scimagojr.xlsx')
    merged_df = ScimEn.iloc[0:15].drop(columns=['Region']).merge(energy, how = 'inner', on = 'Country').merge(GDP_years, how = 'inner', on = 'Country')
    final_df = merged_df.set_index('Country')

    return (final_df)

TOP15 = answer_one()
print(TOP15)

                    Rank  Documents  Citable documents  Citations  \
Country                                                             
China                  1     273437             272374    2336764   
United States          2     175891             172431    2230544   
India                  3      55082              53775     463165   
Japan                  4      50523              50065     488062   
United Kingdom         5      43389              42284     615670   
Germany                6      38739              38013     433148   
Russian Federation     7      36735              36560     115938   
Canada                 8      33472              32863     568080   
Italy                  9      27983              26940     352993   
South Korea           10      27655              27445     328488   
France                11      25232              24732     343860   
Iran                  12      22933              22734     307280   
Spain                 13      2195

In [147]:
def answer_two(df):
    avgGDP = df.iloc[:, -10:].mean(axis = 1, skipna = True)
    avgGDP = avgGDP.to_frame(name = 'Average GDP').sort_values(by = 'Average GDP', ascending = False)

    return avgGDP
avg_country_GDP = answer_two(TOP15)
print(avg_country_GDP)

                     Average GDP
Country                         
United States       1.570403e+13
China               6.927707e+12
Japan               5.239642e+12
Germany             3.523342e+12
United Kingdom      2.780276e+12
France              2.691337e+12
Italy               2.142986e+12
Brazil              1.988889e+12
Russian Federation  1.666746e+12
Canada              1.616359e+12
India               1.602352e+12
Spain               1.400886e+12
South Korea         1.221372e+12
Australia           1.207513e+12
Iran                4.563261e+11


In [148]:
def answer_three(df, country):
    GDP_then = df.loc[country, '2006']
    GDP_now = df.loc[country, '2015']
    GDP_change = (GDP_now - GDP_then)

    return GDP_change

print(answer_three(TOP15, avg_country_GDP.index[5]))

118652421857.7959


In [149]:
def answer_four(df):
    df['Self-citation Ratio'] = df['Self-citations'] / df['Citations']
    max_citation_country = df['Self-citation Ratio'].idxmax()
    max_citation_value = df['Self-citation Ratio'].max()
    result = (max_citation_country, max_citation_value)

    return result

print(answer_four(TOP15))

('China', np.float64(0.6912289816173135))


In [150]:
def answer_five(df):
    df['Estimated population'] = df['Energy Supply'] / df['Energy Supply per Capita']
    global most_populous
    most_populous = df['Estimated population'].sort_values(ascending = False)

    return most_populous.index[3]

print(answer_five(TOP15))

Brazil


In [151]:
def answer_six(df):
    df['Citable docs per Capita'] = df['Citable documents'] / most_populous
    corr = df['Citable docs per Capita'].astype(float).corr(df['Energy Supply per Capita'].astype(float), method='pearson')

    return corr

print(answer_six(TOP15))

0.7434709127726777


In [152]:
def answer_seven(df):
    ContinentDict  = {'China':'Asia',
                  'United States':'North America',
                  'Japan':'Asia',
                  'United Kingdom':'Europe',
                  'Russian Federation':'Europe',
                  'Canada':'North America',
                  'Germany':'Europe',
                  'India':'Asia',
                  'France':'Europe',
                  'South Korea':'Asia',
                  'Italy':'Europe',
                  'Spain':'Europe',
                  'Iran':'Asia',
                  'Australia':'Australia',
                  'Brazil':'South America'}

    df['Estimated population'] = most_populous
    df['Continent'] = df.index.map(ContinentDict)
    grouped = df.groupby('Continent')['Estimated population'].agg(['size', 'sum', 'mean', 'std'])

    return grouped

print(answer_seven(TOP15))


               size               sum              mean           std
Continent                                                            
Asia              5   2898666386.6106   579733277.32212  6.790979e+08
Australia         1   23316017.316017   23316017.316017           NaN
Europe            6  457929667.216372   76321611.202729  3.464767e+07
North America     2   352855249.48025  176427624.740125  1.996696e+08
South America     1  205915254.237288  205915254.237288           NaN
